[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/063.%20The%20Zone%20Picking%20%26%20Pick-and-Pass%20Balancing%20Problem/P63-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 63. The Zone Picking and Pick-and-Pass Balancing Problem
## Tier 3: Artificial Bee Colony Optimization

### Purpose and scope

This notebook uses the Artificial Bee Colony (ABC) metaheuristic to search for a schedule for a set of customer orders. Each order requires picking in one or more specified zones. The algorithm assigns a time slot to each required order-zone operation, then evaluates the resulting schedule using a composite fitness score.

The notebook has four main outputs: the best schedule found, its fitness and diagnostic metrics, plots that track the population during the search, and a parameter-sensitivity experiment. ABC is a stochastic search method, so it does not certify that the best schedule found is globally optimal.

### Modeling assumptions

- The required zones and item counts for each order are fixed input data. The decisions are the time slots assigned to the required order-zone operations; the algorithm does not reassign items to different zones.
- A time slot is an integer-valued scheduling time measured in minutes. The search keeps assigned slots within the planning horizon.
- An order's completion-time proxy is the latest assigned slot plus the processing time of its slowest-starting zone operation, followed by a handoff allowance for each transition between required zones. Zone precedence, simultaneous worker availability, and hard capacity constraints are not enforced.
- Zone workload is measured as the largest amount of processing work assigned to any one time slot in that zone. The balance metric compares these peak slot workloads across zones. This makes the metric responsive to the time-slot decisions, while remaining a proxy rather than a detailed labor-capacity model.
- The supplied zone capacities and worker counts are descriptive data; this model does not use them as capacity constraints. Pick times, item counts, and handoff time are used in the schedule metrics.
- Priorities are retained in the input data but are not weighted in the objective.

These assumptions define a compact search model for illustrating ABC. A detailed operational schedule would also need explicit precedence, capacity, and worker-availability constraints.

### ABC search in this problem

A **food source** is one complete candidate schedule, and its **fitness** measures how desirable that schedule is. The colony repeatedly applies three search phases:

1. **Initialization:** create random schedules and evaluate them.
2. **Employed bees:** each source is locally modified; the new schedule replaces it if its fitness is higher.
3. **Onlooker bees:** select sources with probabilities proportional to fitness, then locally explore the selected schedules using the same greedy acceptance rule.
4. **Scout bees:** replace sources that have failed to improve for more than the abandonment limit.
5. **Tracking:** record fitness and population diversity so that search progress can be inspected.

The employed and onlooker phases exploit promising schedules; the scout phase introduces new random schedules and therefore supports exploration. The update rules are stochastic, and the best-so-far schedule is retained throughout the search.

### Mathematical model and fitness

Let \(O\) be the set of orders, and let \(Z_o\) be the set of zones required by order \(o\). A schedule is \(s = (t_{oz})\), where \(t_{oz}\) is the assigned time slot for order \(o\) in required zone \(z\), with \(t_{oz} \in \{0,\ldots,H-1\}\) and planning horizon \(H\).

For order \(o\), let \(q_{oz}\) be its item count in zone \(z\), and let \(p_z\) be that zone's pick time in minutes per item. The processing time is \(q_{oz}p_z\). The model's order completion-time proxy and total completion-time measure are

$$
C_o(s) = \max_{z \in Z_o}\left(t_{oz} + q_{oz}p_z\right) + h\max\left(|Z_o|-1, 0\right),
\qquad
T(s) = \sum_{o \in O} C_o(s),
$$

where \(h\) is the handoff time in minutes. The handoff allowance is added after the latest zone operation; it is a proxy and does not impose a zone-to-zone precedence schedule.

For each zone \(z\) and slot \(t\), define the assigned workload

$$
L_{zt}(s) = \sum_{o: z \in Z_o,\;t_{oz}=t} q_{oz}p_z,
\qquad
P_z(s) = \max_{0 \leq t < H} L_{zt}(s).
$$

The workload-balance measure is

$$
B(s) = \frac{\max_z P_z(s)-\min_z P_z(s)}{\max_z P_z(s)},
$$

when the denominator is positive, and zero otherwise. Lower values indicate more similar peak slot workloads across zones. This range-based measure is not a statistical variance.

The objective is a weighted composite score to maximize:

$$
F(s) = 0.4\frac{1}{1+T(s)/100}
      + 0.4\frac{1}{1+B(s)}
      + 0.2\min\left(\frac{|O|/\max(T(s),1)}{0.5},1\right).
$$

The three terms reward a lower total completion-time proxy, more balanced peak workloads, and higher throughput proxy, respectively. Their weights and normalization constants are modeling choices in this example. In the code, these quantities are calculated by `_calculate_total_completion_time`, `_calculate_workload_balance`, and `_calculate_fitness`.

### How to interpret the search

Fitness is a score for comparing schedules under this model, not a direct operational service-level measure. A higher value is preferred by the algorithm, but it does not imply a proven optimum or operational feasibility under constraints that the model omits. The convergence plots describe this run and its population; stochastic runs may produce different trajectories and schedules.

The example uses six zones, 25 generated orders, a colony size of 30, and 100 iterations. The data-generation seed is fixed for reproducibility.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Both random-number generators are seeded because the search uses both NumPy and random.
np.random.seed(42)
random.seed(42)

## 1. Define the warehouse data

Each `Zone` supplies its descriptive capacity, pick time, and worker count. Each `Order` contains a fixed item count for every zone it requires and a priority label. The generated order requirements determine which order-zone operations receive time-slot decisions.

The NumPy seed is set in the preceding cell so that the generated order data are repeatable. The Python random generator is seeded as well because the ABC search uses it for selection and local changes.

In [ ]:
@dataclass
class Zone:
    """Descriptive data for one warehouse picking zone."""

    zone_id: int
    capacity: float  # Items per hour; not enforced as a model constraint.
    pick_time: float  # Minutes per item.
    workers: int


@dataclass
class Order:
    """An order with fixed item requirements by zone."""

    order_id: int
    items_by_zone: Dict[int, int]
    priority: int  # 1 = high, 2 = medium, 3 = low; not weighted in this model.


@dataclass
class BeeSolution:
    """One complete schedule and its state in the ABC search."""

    assignments: Dict[int, Dict[int, int]]  # order_id -> zone_id -> time slot
    fitness: float
    trial_count: int = 0  # Consecutive unsuccessful trials since the last improvement.


zones = [
    Zone(1, 180, 1.8, 3),
    Zone(2, 150, 2.2, 2),
    Zone(3, 200, 1.5, 3),
    Zone(4, 140, 2.8, 2),
    Zone(5, 170, 2.0, 2),
    Zone(6, 160, 1.6, 3),
]

orders = []
for order_id in range(1, 26):
    items_by_zone = {}
    number_of_zones = np.random.randint(2, 5)
    selected_zones = np.random.choice(range(1, 7), number_of_zones, replace=False)

    for zone_id in selected_zones:
        items_by_zone[int(zone_id)] = int(np.random.randint(1, 10))

    priority = int(np.random.choice([1, 2, 3], p=[0.3, 0.5, 0.2]))
    orders.append(Order(order_id, items_by_zone, priority))

handoff_time = 2  # Minutes per zone transition in the completion-time proxy.
time_horizon = 30  # Number of available integer-valued time slots.

print("Zone configuration:")
for zone in zones:
    print(
        f"  Zone {zone.zone_id}: {zone.capacity} items/hour, "
        f"{zone.pick_time} minutes/item, {zone.workers} workers"
    )

print("\nOrder summary (first 10 shown):")
for order in orders[:10]:
    priority_label = {1: "High", 2: "Medium", 3: "Low"}[order.priority]
    print(f"  Order {order.order_id} ({priority_label} priority): {order.items_by_zone}")

print(f"\nTotal orders: {len(orders)}")

## 2. Implement the Artificial Bee Colony search

A population member stores the schedule as a nested dictionary: the outer key is an order ID, and the inner mapping assigns each required zone a time slot. A **neighbor** is formed by selecting one order-zone assignment and changing its slot while keeping it within the planning horizon.

The ABC phases preserve the method's central sequence: initialize food sources, apply employed-bee local search, select food sources probabilistically for onlooker-bee search, and replace exhausted sources through scout search. Greedy acceptance keeps a neighbor only when its fitness is strictly higher. The best schedule found is stored separately so that later scout replacements cannot discard it.

Population diversity is the mean normalized Hamming distance across all pairs of schedules. Two schedules differ at an assignment when they give that order-zone operation different integer time slots. This statistic ranges from zero (identical schedules) to one (every compared assignment differs).

In [ ]:
class ArtificialBeeColonyOptimizer:
    """Artificial Bee Colony search for the time-slot schedule model."""

    def __init__(
        self,
        zones: List[Zone],
        orders: List[Order],
        colony_size: int = 30,
        max_iterations: int = 100,
        limit: int = 10,
        handoff_time: float = 2,
        time_horizon: int = 30,
    ):
        if not zones:
            raise ValueError("At least one zone is required.")
        if not orders:
            raise ValueError("At least one order is required.")
        if colony_size < 1 or max_iterations < 1 or time_horizon < 1:
            raise ValueError("Colony size, iteration count, and horizon must be positive.")
        if limit < 0:
            raise ValueError("The abandonment limit cannot be negative.")

        self.zones = zones
        self.orders = orders
        self.colony_size = colony_size
        self.max_iterations = max_iterations
        self.limit = limit
        self.handoff_time = handoff_time
        self.time_horizon = time_horizon
        self.zone_by_id = {zone.zone_id: zone for zone in zones}

        missing_zone_ids = {
            zone_id
            for order in orders
            for zone_id in order.items_by_zone
            if zone_id not in self.zone_by_id
        }
        if missing_zone_ids:
            raise ValueError(f"Orders refer to unknown zones: {sorted(missing_zone_ids)}")

        self.food_sources: List[BeeSolution] = []
        self.best_solution: Optional[Dict[int, Dict[int, int]]] = None
        self.best_fitness = -float("inf")
        self.fitness_history = []
        self.diversity_history = []
        self.convergence_data = []
        self.scout_activations = 0

    def optimize(self) -> Dict:
        """Run initialization followed by the three ABC phases per iteration."""
        start_time = time.time()
        self._initialize_population()

        for iteration in range(self.max_iterations):
            self._employed_bee_phase()
            self._onlooker_bee_phase()
            self._scout_bee_phase()
            self._update_best_solution()
            self._track_convergence(iteration)

            if iteration % 20 == 0:
                average_fitness = np.mean([source.fitness for source in self.food_sources])
                print(
                    f"Iteration {iteration}: best fitness = {self.best_fitness:.4f}, "
                    f"population average = {average_fitness:.4f}"
                )

        return {
            "best_solution": self.best_solution,
            "best_fitness": self.best_fitness,
            "fitness_history": self.fitness_history,
            "diversity_history": self.diversity_history,
            "convergence_data": self.convergence_data,
            "optimization_time": time.time() - start_time,
            "final_population": self.food_sources,
            "scout_activations": self.scout_activations,
        }

    def _initialize_population(self) -> None:
        """Create and evaluate the initial random food sources."""
        self.food_sources = []
        self.best_solution = None
        self.best_fitness = -float("inf")
        self.fitness_history = []
        self.diversity_history = []
        self.convergence_data = []
        self.scout_activations = 0

        for _ in range(self.colony_size):
            solution = self._generate_random_solution()
            fitness = self._calculate_fitness(solution)
            self.food_sources.append(BeeSolution(solution, fitness))

            if fitness > self.best_fitness:
                self.best_fitness = fitness
                self.best_solution = copy.deepcopy(solution)

    def _generate_random_solution(self) -> Dict[int, Dict[int, int]]:
        """Assign every required order-zone operation a slot in the horizon."""
        solution = {}
        for order in self.orders:
            solution[order.order_id] = {
                zone_id: int(np.random.randint(0, self.time_horizon))
                for zone_id in sorted(order.items_by_zone)
            }
        return solution

    def _calculate_fitness(self, solution: Dict[int, Dict[int, int]]) -> float:
        """Combine completion time, peak workload balance, and throughput proxies."""
        total_completion_time = self._calculate_total_completion_time(solution)
        workload_balance = self._calculate_workload_balance(solution)
        throughput = len(self.orders) / max(total_completion_time, 1)

        time_fitness = 1 / (1 + total_completion_time / 100)
        balance_fitness = 1 / (1 + workload_balance)
        throughput_fitness = min(throughput / 0.5, 1)

        return (
            0.4 * time_fitness
            + 0.4 * balance_fitness
            + 0.2 * throughput_fitness
        )

    def _calculate_total_completion_time(
        self, solution: Dict[int, Dict[int, int]]
    ) -> float:
        """Sum the modeled completion-time proxy over the orders."""
        total_time = 0.0

        for order in self.orders:
            assignments = solution.get(order.order_id, {})
            if not assignments:
                continue

            zone_completion_times = []
            for zone_id, time_slot in assignments.items():
                if zone_id not in order.items_by_zone:
                    continue
                zone = self.zone_by_id[zone_id]
                processing_time = order.items_by_zone[zone_id] * zone.pick_time
                zone_completion_times.append(time_slot + processing_time)

            if zone_completion_times:
                handoff_allowance = self.handoff_time * max(
                    len(order.items_by_zone) - 1, 0
                )
                total_time += max(zone_completion_times) + handoff_allowance

        return total_time

    def _calculate_workload_balance(
        self, solution: Dict[int, Dict[int, int]]
    ) -> float:
        """Compare zones' peak assigned work across the available time slots."""
        slot_loads = {
            zone.zone_id: defaultdict(float)
            for zone in self.zones
        }

        for order in self.orders:
            assignments = solution.get(order.order_id, {})
            for zone_id, time_slot in assignments.items():
                if zone_id not in order.items_by_zone or zone_id not in slot_loads:
                    continue
                zone = self.zone_by_id[zone_id]
                processing_time = order.items_by_zone[zone_id] * zone.pick_time
                slot_loads[zone_id][time_slot] += processing_time

        peak_loads = [
            max(loads.values(), default=0.0)
            for loads in slot_loads.values()
        ]
        if not peak_loads or max(peak_loads) == 0:
            return 0.0

        return (max(peak_loads) - min(peak_loads)) / max(peak_loads)

    def _employed_bee_phase(self) -> None:
        """Locally explore every food source and accept improving neighbors."""
        for source in self.food_sources:
            neighbor = self._generate_neighbor_solution(source.assignments)
            neighbor_fitness = self._calculate_fitness(neighbor)

            if neighbor_fitness > source.fitness:
                source.assignments = neighbor
                source.fitness = neighbor_fitness
                source.trial_count = 0
            else:
                source.trial_count += 1

    def _generate_neighbor_solution(
        self, current_solution: Dict[int, Dict[int, int]]
    ) -> Dict[int, Dict[int, int]]:
        """Change one assignment by a small amount while respecting the horizon."""
        neighbor = copy.deepcopy(current_solution)
        eligible_orders = [
            order_id for order_id, assignments in neighbor.items() if assignments
        ]
        if not eligible_orders:
            return neighbor

        order_id = random.choice(eligible_orders)
        zone_id = random.choice(list(neighbor[order_id]))
        current_time = neighbor[order_id][zone_id]
        new_time = current_time + random.randint(-3, 3)
        neighbor[order_id][zone_id] = max(
            0, min(self.time_horizon - 1, new_time)
        )
        return neighbor

    def _onlooker_bee_phase(self) -> None:
        """Select sources in proportion to fitness and search their neighbors."""
        fitness_values = [source.fitness for source in self.food_sources]
        total_fitness = sum(fitness_values)

        if total_fitness > 0:
            probabilities = [fitness / total_fitness for fitness in fitness_values]
        else:
            probabilities = [1 / len(self.food_sources)] * len(self.food_sources)

        for _ in range(len(self.food_sources)):
            selected_index = self._roulette_wheel_selection(probabilities)
            source = self.food_sources[selected_index]
            neighbor = self._generate_neighbor_solution(source.assignments)
            neighbor_fitness = self._calculate_fitness(neighbor)

            if neighbor_fitness > source.fitness:
                source.assignments = neighbor
                source.fitness = neighbor_fitness
                source.trial_count = 0
            else:
                source.trial_count += 1

    def _roulette_wheel_selection(self, probabilities: List[float]) -> int:
        """Sample one food source using cumulative fitness probabilities."""
        draw = random.random()
        cumulative_probability = 0.0

        for index, probability in enumerate(probabilities):
            cumulative_probability += probability
            if draw <= cumulative_probability:
                return index

        return len(probabilities) - 1

    def _scout_bee_phase(self) -> None:
        """Replace a source after its unsuccessful-trial count exceeds the limit."""
        for index, source in enumerate(self.food_sources):
            if source.trial_count > self.limit:
                solution = self._generate_random_solution()
                fitness = self._calculate_fitness(solution)
                self.food_sources[index] = BeeSolution(solution, fitness)
                self.scout_activations += 1

    def _update_best_solution(self) -> None:
        """Retain the best schedule found, including across scout replacements."""
        for source in self.food_sources:
            if source.fitness > self.best_fitness:
                self.best_fitness = source.fitness
                self.best_solution = copy.deepcopy(source.assignments)

    def _track_convergence(self, iteration: int) -> None:
        """Record population fitness statistics and schedule diversity."""
        fitness_values = [source.fitness for source in self.food_sources]
        diversity = self._calculate_population_diversity()

        self.fitness_history.append(
            {
                "iteration": iteration,
                "best": max(fitness_values),
                "average": float(np.mean(fitness_values)),
                "worst": min(fitness_values),
                "std": float(np.std(fitness_values)),
            }
        )
        self.diversity_history.append(
            {"iteration": iteration, "diversity": diversity}
        )
        self.convergence_data.append(
            {
                "iteration": iteration,
                "best_fitness": self.best_fitness,
                "avg_fitness": float(np.mean(fitness_values)),
                "diversity": diversity,
            }
        )

    def _calculate_population_diversity(self) -> float:
        """Average pairwise normalized Hamming distance in the population."""
        if len(self.food_sources) < 2:
            return 0.0

        distances = []
        for first_index in range(len(self.food_sources)):
            for second_index in range(first_index + 1, len(self.food_sources)):
                distances.append(
                    self._hamming_distance(
                        self.food_sources[first_index].assignments,
                        self.food_sources[second_index].assignments,
                    )
                )

        return float(np.mean(distances)) if distances else 0.0

    def _hamming_distance(self, sol1: Dict, sol2: Dict) -> float:
        """Return the fraction of shared assignments with different time slots."""
        differences = 0
        comparisons = 0

        for order_id, assignments in sol1.items():
            other_assignments = sol2.get(order_id, {})
            for zone_id, time_slot in assignments.items():
                if zone_id in other_assignments:
                    differences += time_slot != other_assignments[zone_id]
                    comparisons += 1

        return differences / comparisons if comparisons else 0.0

## 3. Run the ABC optimizer and inspect its schedule

The optimizer uses the example's 30-source colony, 100 iterations, abandonment limit of 10, two-minute handoff allowance, and 30-slot horizon. The returned `best_solution` is the best schedule encountered during the run; `best_fitness` is its composite score.

The diagnostic values below are computed from that same schedule. The reported workload-balance value compares peak assigned work by zone, while the printed per-zone peaks show the quantities used in that comparison. These are model diagnostics, not capacity utilization percentages.

In [ ]:
abc_optimizer = ArtificialBeeColonyOptimizer(
    zones=zones,
    orders=orders,
    colony_size=30,
    max_iterations=100,
    limit=10,
    handoff_time=handoff_time,
    time_horizon=time_horizon,
)

print("Starting Artificial Bee Colony optimization...")
result = abc_optimizer.optimize()

best_solution = result["best_solution"]
if best_solution is None:
    raise RuntimeError("The optimizer did not produce a schedule.")

assert len(best_solution) == len(orders)
for order in orders:
    assert set(best_solution[order.order_id]) == set(order.items_by_zone)
    assert all(
        0 <= time_slot < time_horizon
        for time_slot in best_solution[order.order_id].values()
    )

completion_time = abc_optimizer._calculate_total_completion_time(best_solution)
workload_balance = abc_optimizer._calculate_workload_balance(best_solution)
throughput = len(orders) / max(completion_time, 1)
recomputed_fitness = abc_optimizer._calculate_fitness(best_solution)
assert np.isclose(recomputed_fitness, result["best_fitness"])

print(f"\nOptimization time: {result['optimization_time']:.2f} seconds")
print(f"Best fitness: {result['best_fitness']:.4f}")
print(f"Total completion-time proxy: {completion_time:.1f} minutes")
print(f"Peak-workload balance range: {workload_balance:.3f}")
print(f"Throughput proxy: {throughput:.3f} orders per summed completion minute")
print(f"Scout replacements during the run: {result['scout_activations']}")

print("\nPeak assigned workload by zone:")
for zone in zones:
    zone_peak = 0.0
    for order in orders:
        time_slot = best_solution[order.order_id].get(zone.zone_id)
        if time_slot is not None:
            zone_peak += 0  # The slot-specific accumulation is computed below.

    slot_workloads = defaultdict(float)
    for order in orders:
        time_slot = best_solution[order.order_id].get(zone.zone_id)
        if time_slot is not None:
            slot_workloads[time_slot] += (
                order.items_by_zone[zone.zone_id] * zone.pick_time
            )
    zone_peak = max(slot_workloads.values(), default=0.0)
    print(f"  Zone {zone.zone_id}: {zone_peak:.2f} processing minutes in one slot")

## 4. Examine convergence and population diversity

The first plot tracks the best, average, and worst fitness among the current food sources at each iteration. The shaded band emphasizes the spread between the average and worst values. The second plot shows average pairwise schedule diversity; a decline can indicate that the population is becoming more similar, although scout replacements can introduce new variation. The third plot reports fitness standard deviation, another measure of population spread.

The final panel shows the relative change in the current population's best fitness from one recorded iteration to the next. The separately retained best-so-far fitness cannot decrease, but the current population's best can change after scout replacement. Therefore, a negative relative change in this panel is possible and is not evidence that the stored best schedule was lost.

A plotted trend or a small improvement rate is descriptive, not a proof of convergence or optimality.

In [ ]:
def visualize_abc_convergence(result: Dict) -> None:
    """Plot fitness, diversity, spread, and iteration-to-iteration change."""
    fitness_history = result["fitness_history"]
    diversity_history = result["diversity_history"]
    if not fitness_history:
        raise ValueError("Convergence history is empty.")

    iterations = [entry["iteration"] for entry in fitness_history]
    best_fitness = [entry["best"] for entry in fitness_history]
    average_fitness = [entry["average"] for entry in fitness_history]
    worst_fitness = [entry["worst"] for entry in fitness_history]
    fitness_std = [entry["std"] for entry in fitness_history]
    diversity = [entry["diversity"] for entry in diversity_history]

    figure, axes = plt.subplots(2, 2, figsize=(16, 12))
    figure.suptitle("Artificial Bee Colony Search Diagnostics", fontsize=16)

    axes[0, 0].plot(iterations, best_fitness, color="green", label="Best in population")
    axes[0, 0].plot(iterations, average_fitness, color="blue", label="Population average")
    axes[0, 0].plot(iterations, worst_fitness, color="red", label="Worst in population")
    axes[0, 0].fill_between(
        iterations, average_fitness, worst_fitness, alpha=0.2, color="lightblue"
    )
    axes[0, 0].set_xlabel("Iteration")
    axes[0, 0].set_ylabel("Fitness (unitless)")
    axes[0, 0].set_title("Population fitness")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(iterations, diversity, color="purple", marker="o", markersize=3)
    axes[0, 1].set_xlabel("Iteration")
    axes[0, 1].set_ylabel("Mean normalized Hamming distance")
    axes[0, 1].set_title("Schedule diversity")
    axes[0, 1].grid(True, alpha=0.3)
    if len(iterations) > 1:
        trend = np.poly1d(np.polyfit(iterations, diversity, 1))
        axes[0, 1].plot(iterations, trend(iterations), "r--", label="Linear trend")
        axes[0, 1].legend()

    axes[1, 0].plot(iterations, fitness_std, color="orange", marker="s", markersize=3)
    axes[1, 0].set_xlabel("Iteration")
    axes[1, 0].set_ylabel("Fitness standard deviation")
    axes[1, 0].set_title("Population fitness spread")
    axes[1, 0].grid(True, alpha=0.3)

    improvement_rates = []
    for previous, current in zip(best_fitness[:-1], best_fitness[1:]):
        if np.isclose(previous, 0.0):
            improvement_rates.append(0.0)
        else:
            improvement_rates.append((current - previous) / abs(previous))

    rate_iterations = iterations[1:]
    axes[1, 1].plot(rate_iterations, improvement_rates, color="teal", marker="^", markersize=3)
    axes[1, 1].axhline(0, color="red", linestyle="--", alpha=0.7)
    axes[1, 1].set_xlabel("Iteration")
    axes[1, 1].set_ylabel("Relative change in population-best fitness")
    axes[1, 1].set_title("Iteration-to-iteration change")
    axes[1, 1].grid(True, alpha=0.3)

    figure.tight_layout()
    plt.show()

    initial_best = best_fitness[0]
    final_best = best_fitness[-1]
    improvement = final_best - initial_best
    improvement_percent = (
        100 * improvement / abs(initial_best)
        if not np.isclose(initial_best, 0.0)
        else float("nan")
    )

    convergence_iteration = None
    for index in range(9, len(improvement_rates)):
        recent_changes = improvement_rates[index - 9 : index + 1]
        if all(abs(rate) < 0.01 for rate in recent_changes):
            convergence_iteration = rate_iterations[index]
            break

    print("\nABC convergence summary")
    print(f"Initial population-best fitness: {initial_best:.4f}")
    print(f"Final population-best fitness: {final_best:.4f}")
    print(f"Change in population-best fitness: {improvement:.4f}")
    if np.isfinite(improvement_percent):
        print(f"Relative change from the initial value: {improvement_percent:.1f}%")
    else:
        print("Relative change is undefined because the initial value is zero.")

    if convergence_iteration is None:
        print("No 10-iteration interval met the descriptive 1% change criterion.")
    else:
        print(
            "First interval meeting the descriptive 1% change criterion ends at "
            f"iteration {convergence_iteration}."
        )

    print(f"Initial population diversity: {diversity[0]:.4f}")
    print(f"Final population diversity: {diversity[-1]:.4f}")


visualize_abc_convergence(result)

## 5. Explore parameter sensitivity

The following experiment varies colony size and the scout abandonment limit while holding the data and iteration count fixed. Colony size controls how many food sources are searched; the limit controls how many unsuccessful trials a source can accumulate before scout replacement.

To keep every test internally consistent, each experiment uses the complete set of zones and orders. In particular, an order is never evaluated against a zone list that omits one of its required zones. The reported scout count is the number of actual source replacements during the run, rather than the number of final sources whose trial counter happens to be zero.

Each setting is a stochastic run. The outputs are observations for this generated instance, not general guarantees about the effect of a parameter.

In [ ]:
def parameter_sensitivity_analysis() -> Tuple[List[Dict], List[Dict]]:
    """Measure how colony size and abandonment limit affect ABC runs."""
    colony_sizes = [10, 20, 30, 40, 50]
    colony_results = []

    print("Colony-size experiments")
    print(f"{'Colony size':<14} {'Best fitness':<14} {'Time (s)':<12} {'Diversity':<12}")
    print("-" * 52)

    for size in colony_sizes:
        optimizer = ArtificialBeeColonyOptimizer(
            zones=zones,
            orders=orders,
            colony_size=size,
            max_iterations=50,
            limit=10,
            handoff_time=handoff_time,
            time_horizon=time_horizon,
        )
        test_result = optimizer.optimize()
        final_diversity = test_result["diversity_history"][-1]["diversity"]
        colony_results.append(
            {
                "colony_size": size,
                "best_fitness": test_result["best_fitness"],
                "time": test_result["optimization_time"],
                "diversity": final_diversity,
            }
        )
        print(
            f"{size:<14} {test_result['best_fitness']:<14.4f} "
            f"{test_result['optimization_time']:<12.2f} {final_diversity:<12.4f}"
        )

    limit_values = [5, 10, 15, 20, 25]
    limit_results = []

    print("\nAbandonment-limit experiments")
    print(f"{'Limit':<10} {'Best fitness':<14} {'Time (s)':<12} {'Scout replacements':<20}")
    print("-" * 58)

    for limit_value in limit_values:
        optimizer = ArtificialBeeColonyOptimizer(
            zones=zones,
            orders=orders,
            colony_size=20,
            max_iterations=50,
            limit=limit_value,
            handoff_time=handoff_time,
            time_horizon=time_horizon,
        )
        test_result = optimizer.optimize()
        limit_results.append(
            {
                "limit": limit_value,
                "best_fitness": test_result["best_fitness"],
                "time": test_result["optimization_time"],
                "scout_replacements": test_result["scout_activations"],
            }
        )
        print(
            f"{limit_value:<10} {test_result['best_fitness']:<14.4f} "
            f"{test_result['optimization_time']:<12.2f} "
            f"{test_result['scout_activations']:<20}"
        )

    figure, axes = plt.subplots(2, 2, figsize=(14, 10))
    figure.suptitle("ABC Parameter Sensitivity", fontsize=16)

    axes[0, 0].plot(
        [entry["colony_size"] for entry in colony_results],
        [entry["best_fitness"] for entry in colony_results],
        "o-",
        color="blue",
    )
    axes[0, 0].set_xlabel("Colony size (food sources)")
    axes[0, 0].set_ylabel("Best fitness (unitless)")
    axes[0, 0].set_title("Colony size and solution score")
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(
        [entry["colony_size"] for entry in colony_results],
        [entry["time"] for entry in colony_results],
        "s-",
        color="red",
    )
    axes[0, 1].set_xlabel("Colony size (food sources)")
    axes[0, 1].set_ylabel("Optimization time (seconds)")
    axes[0, 1].set_title("Colony size and run time")
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(
        [entry["limit"] for entry in limit_results],
        [entry["best_fitness"] for entry in limit_results],
        "o-",
        color="green",
    )
    axes[1, 0].set_xlabel("Abandonment limit (trials)")
    axes[1, 0].set_ylabel("Best fitness (unitless)")
    axes[1, 0].set_title("Abandonment limit and solution score")
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(
        [entry["limit"] for entry in limit_results],
        [entry["scout_replacements"] for entry in limit_results],
        "s-",
        color="purple",
    )
    axes[1, 1].set_xlabel("Abandonment limit (trials)")
    axes[1, 1].set_ylabel("Scout replacements (count)")
    axes[1, 1].set_title("Abandonment limit and scout activity")
    axes[1, 1].grid(True, alpha=0.3)

    figure.tight_layout()
    plt.show()
    return colony_results, limit_results


colony_results, limit_results = parameter_sensitivity_analysis()

## 6. Inspect the best schedule by order

This diagnostic view makes the decision variables visible: for every order, it lists each required zone, the assigned slot, and the processing time implied by the order's item count and the zone's pick time. The table does not add constraints to the model; it helps connect the returned schedule to the input data and the completion-time proxy.

In [ ]:
schedule_rows = []
for order in orders:
    for zone_id, time_slot in sorted(best_solution[order.order_id].items()):
        zone = abc_optimizer.zone_by_id[zone_id]
        item_count = order.items_by_zone[zone_id]
        schedule_rows.append(
            {
                "Order": order.order_id,
                "Priority": {1: "High", 2: "Medium", 3: "Low"}[order.priority],
                "Zone": zone_id,
                "Items": item_count,
                "Assigned slot (minutes)": time_slot,
                "Processing time (minutes)": item_count * zone.pick_time,
            }
        )

schedule_table = pd.DataFrame(schedule_rows)
print("Best-so-far schedule, sorted by order and zone:")
display(schedule_table)

## 7. Interpretation and limitations

ABC is useful here as a population-based search procedure: multiple candidate schedules are explored, promising neighbors are retained greedily, and scout replacements help renew search when food sources stop improving. The fitness history and diversity plot provide complementary views of search behavior; neither alone establishes solution quality beyond the defined score.

The objective combines completion-time, peak-workload-balance, and throughput proxies. Because it is a weighted sum, improving one component can be offset by another, and the normalization constants affect the relative influence of the components. The sensitivity experiment illustrates how selected algorithm parameters behave on this instance; it does not establish universal parameter settings.

The schedule is not a capacity-feasible warehouse plan. Capacity, worker availability, zone precedence, item travel, and operational constraints are not enforced, and priority does not affect the score. The handoff allowance contributes to the completion-time proxy but is not represented as a sequence of intermediate handoff events. The model should therefore be interpreted as an instructional ABC search example, not as a validated dispatch plan.

The method has no optimality guarantee. Different random runs can return different schedules, and a stronger fitness score means only that the schedule is preferred by the specified objective. Natural extensions within this modeling framework include adding operational constraints and testing larger instances; those extensions require explicit model definitions and are not implemented here.